# ISO 19005 PDF/A Compliance for European & Latin American E-Government Portals
**Technical Architecture, veraPDF Conformance Profiles, Hybrid Invoicing (FacturaE, ZUGFeRD, Factur-X) & Long-Term Archival**

*Published by the [KaijuConverter Engineering Research Group](https://kaijuconverter.com)*

Public administrations across the European Union (under the eIDAS regulation, the Single Digital Gateway, and national interoperability frameworks) and Latin America (SAT Mexico, DIAN Colombia, SEFAZ Brazil) enforce strict digital preservation standards for electronic administrative files, judicial dossiers, and commercial invoicing. Standard PDF documents generated by desktop software are routinely rejected by official validation gateways due to uncalibrated RGB color profiles, non-embedded proprietary typography, embedded JavaScript scripts, and missing XMP metadata.

This engineering notebook analyzes the technical requirements of the **ISO 19005** specification family (PDF/A-1, PDF/A-2, PDF/A-3), benchmarks automated Ghostscript and veraPDF validation pipelines, and details the hybrid XML embedding architecture mandated by European e-invoicing standards.

### Official Multi-Language Converters & Archival Utilities
- **Global PDF/A Engine**: [Online PDF to PDF/A Long-Term Archival Converter](https://kaijuconverter.com/tools/pdf-to-pdfa)
- **Español (ES)**: [Convertidor PDF a PDF/A para Sedes Electrónicas](https://kaijuconverter.com/es/tools/pdf-to-pdfa)
- **Deutsch (DE)**: [PDF in PDF/A Langzeitarchivierung Konverter (ZUGFeRD & XRechnung)](https://kaijuconverter.com/de/tools/pdf-to-pdfa)
- **Français (FR)**: [Convertisseur PDF en PDF/A Conforme Chorus Pro & Factur-X](https://kaijuconverter.com/fr/tools/pdf-to-pdfa)
- **Italiano (IT)**: [Convertitore PDF/A per Conservazione Digitale Sostitutiva (AGID)](https://kaijuconverter.com/it/tools/pdf-to-pdfa)
- **Guía Técnica de Optimización**: [Cómo Reducir Tamaño de PDF Sin Perder Calidad ni Validez Legal](https://kaijuconverter.com/es/guides/reducir-tamano-pdf-sin-perder-calidad)
- **Ecosystem Tools**: [Lossless PDF Compressor](https://kaijuconverter.com/tools/compress-pdf) | [Side-by-Side Comparison: KaijuConverter vs iLovePDF](https://kaijuconverter.com/compare/kaijuconverter-vs-ilovepdf) | [KaijuConverter vs CloudConvert](https://kaijuconverter.com/compare/kaijuconverter-vs-cloudconvert)


## 1. Regulatory Frameworks: Spain, Germany, France, Italy & Latin America

Government interoperability portals establish deterministic file acceptance criteria based on national technical norms derived from ISO 19005:

| Jurisdiction | Regulatory Body / Portal | Standard Mandated | Invoicing / Document Profile | Common Ingestion Rejection Causes |
|:---|:---|:---|:---|:---|
| **Spain (ES)** | *Esquema Nacional de Interoperabilidad (ENI)*, LexNET, Red SARA | **PDF/A-1b** / **PDF/A-2b** | FacturaE (XML signature attached or hybrid) | Unembedded Arial/Helvetica fonts, dynamic XFA forms, active JavaScript |
| **Germany (DE)** | *KoSIT*, Bundesverwaltung, ZUGFeRD 2.2 / XRechnung | **PDF/A-3b** | Factur-X / ZUGFeRD (`factur-x.xml` embedded in `/AF`) | Missing `AFRelationship/Alternative` tag, non-PDF/A compliant base container |
| **France (FR)** | *Chorus Pro*, AIFE, Direction Générale des Finances Publiques | **PDF/A-3b** | Factur-X (hybrid PDF+XML EN 16931) | Missing XMP extension schema declaration, device RGB unlinked to OutputIntent |
| **Italy (IT)** | *Agenzia per l'Italia Digitale (AGID)*, SDI (Sistema di Interscambio) | **PDF/A-1a** or **PDF/A-2b** | Conservazione Sostitutiva a Norma | Lack of Unicode CMap (`ToUnicode` table), corrupted structural tree tags |
| **LatAm (MX/CO)**| *SAT Mexico* (CFDI), *DIAN Colombia* (Factura Electrónica) | **PDF/A-1b** | Timbre Fiscal Digital (SAT QR + UUID metadata) | Missing XMP Dublin Core metadata, rasterized barcodes below 300 DPI |


## 2. ISO 19005 Conformance Matrix: Level B vs Level U vs Level A

The PDF/A standard is stratified into four major release parts (ISO 19005-1 through 19005-4) and three conformance levels:

1. **Level b (Basic Visual Integrity)**:
   Guarantees that the document renders identically across operating systems and printing hardware 50+ years into the future. Mandates 100% font embedding and standardized color management via `OutputIntent` dictionaries.
2. **Level u (Unicode Searchability)**:
   Extends Level b by requiring exact Unicode mappings (`/ToUnicode` CMap streams) for every glyph. Enables deterministic copy-paste, text indexing, and screen reader accessibility.
3. **Level a (Full Accessibility & Structural Semantics)**:
   Mandates logical document tagging (`/StructTreeRoot`), alternative text for images (`/Alt`), natural language declarations (`/Lang`), and reading order hierarchy according to WCAG 2.1 / PDF/UA standards.

```
+-----------------------------------------------------------------------------------+
|                          ISO 19005 Conformance Levels                             |
+-----------------------------------------------------------------------------------+
| PDF/A-1 (ISO 19005-1:2005) - Based on PDF 1.4                                     |
|   [Level 1b: Basic Visual Conformance]   <--- Most common for legal scans         |
|   [Level 1a: Complete Tagged Accessibility]                                       |
+-----------------------------------------------------------------------------------+
| PDF/A-2 (ISO 19005-2:2011) - Based on PDF 1.7 (ISO 32000-1)                      |
|   [Level 2b: Visual Conformance + JPEG 2000, Transparencies, PDF/A nesting]       |
|   [Level 2u: Unicode Text Conformance]                                            |
|   [Level 2a: Accessible Tagged Structure]                                         |
+-----------------------------------------------------------------------------------+
| PDF/A-3 (ISO 19005-3:2012) - Based on PDF 1.7 (ISO 32000-1)                      |
|   [Level 3b: Visual Conformance + Arbitrary File Embedding in Catalog /AF]        |
|              ---> REQUIRED BY: ZUGFeRD 2.x, Factur-X, Chorus Pro                 |
+-----------------------------------------------------------------------------------+
```


In [1]:
import sys

def audit_pdfa_compliance(document_dict):
    """
    Simulates veraPDF / ISO 19005 algorithmic rule checks on parsed PDF object graph.
    """
    print("=" * 88)
    print("KAIJUCONVERTER ISO 19005 VERIFIER & PRE-FLIGHT AUDIT SIMULATOR")
    print("Testing Compliance: Factur-X / ZUGFeRD 2.2 / BOE Sedes Electrónicas (PDF/A-3b Target)")
    print("=" * 88)
    
    # Check 1: Base Version
    v = document_dict.get("version", "1.4")
    print(f"[CHECK 01] PDF Base Version: PDF {v} ......................................... [PASS]")
    
    # Check 2: OutputIntents
    oi = document_dict.get("output_intents", {})
    if oi.get("present"):
        print(f"[CHECK 02] Document Catalog /OutputIntents Stream Presence .................. [PASS]")
        print(f"           OutputConditionIdentifier: '{oi.get('identifier')}'")
        print(f"           DestOutputProfile ICC Token: Embedded (Size: {oi.get('icc_bytes')} bytes)")
    
    # Check 3: Forbidden tokens
    print("[CHECK 03] Prohibited Object Audit:")
    for forbidden in ["JavaScript Actions (/JS, /JavaScript)", "Encryption & Security Filters (/Encrypt)",
                      "Multimedia / Embedded Sound & Movie (/Movie, /Sound)", "Dynamic XFA Architecture Streams",
                      "LZW Compression Filter Usage"]:
        print(f"           - {forbidden.ljust(50)} [NONE DETECTED - PASS]")
        
    # Check 4: Font embedding
    fonts = document_dict.get("fonts", [])
    print(f"[CHECK 04] Typography & Font Dictionary Audit:")
    print(f"           - Embedded Fonts: {len(fonts)} / {len(fonts)} (100% Embedded Subsets) .................. [PASS]")
    for font in fonts:
        print(f"             * '{font['name']}' ({font['type']}) -> /ToUnicode [{'PRESENT' if font['unicode'] else 'MISSING'}]")
        
    # Check 5: XMP
    xmp = document_dict.get("xmp", {})
    print(f"[CHECK 05] XMP Digital Preservation Metadata Validation:")
    print(f"           - Schema 'http://www.aiim.org/pdfa/ns/id/' ....................... [PASS]")
    print(f"             pdfaid:part = '{xmp.get('part')}', pdfaid:conformance = '{xmp.get('conformance')}'")
    
    # Check 6: Associated Files (PDF/A-3)
    af = document_dict.get("associated_files", {})
    print(f"[CHECK 06] PDF/A-3 Associated Files (/AF) Hybrid XML Audit:")
    print(f"           - Embedded File: '{af.get('filename')}' (MIME: '{af.get('mime')}') ............... [PASS]")
    print(f"           - AFRelationship: '{af.get('relationship')}' ................................. [PASS]")
    print("-" * 88)
    print("OVERALL VERAPDF CONFORMANCE RESULT: 100% COMPLIANT (PDF/A-3b ISO 19005-3:2012)")
    print("Ingestion Status for Chorus Pro (FR) / KoSIT (DE) / BOE Sede (ES): READY FOR SUBMISSION")
    print("=" * 88)

mock_doc = {
    "version": "1.7",
    "output_intents": {"present": True, "identifier": "sRGB IEC61966-2.1", "icc_bytes": 3144},
    "fonts": [
        {"name": "BCDFEE+LiberationSans-Bold", "type": "TrueType subset", "unicode": True},
        {"name": "ACDFEF+LiberationSans-Regular", "type": "TrueType subset", "unicode": True},
        {"name": "ZDFEFA+Courier-Oblique", "type": "CFF subset", "unicode": True},
    ],
    "xmp": {"part": "3", "conformance": "B"},
    "associated_files": {"filename": "factur-x.xml", "mime": "text/xml", "relationship": "/Alternative"}
}
audit_pdfa_compliance(mock_doc)


KAIJUCONVERTER ISO 19005 VERIFIER & PRE-FLIGHT AUDIT SIMULATOR
Testing Compliance: Factur-X / ZUGFeRD 2.2 / BOE Sedes Electrónicas (PDF/A-3b Target)
[CHECK 01] PDF Base Version: PDF 1.7 ......................................... [PASS]
[CHECK 02] Document Catalog /OutputIntents Stream Presence .................. [PASS]
           OutputConditionIdentifier: 'sRGB IEC61966-2.1'
           DestOutputProfile ICC Token: Embedded (Size: 3144 bytes)
[CHECK 03] Prohibited Object Audit:
           - JavaScript Actions (/JS, /JavaScript) .......................... [NONE DETECTED - PASS]
           - Encryption & Security Filters (/Encrypt) ....................... [NONE DETECTED - PASS]
           - Multimedia / Embedded Sound & Movie (/Movie, /Sound) ........... [NONE DETECTED - PASS]
           - Dynamic XFA Architecture Streams ............................... [NONE DETECTED - PASS]
           - LZW Compression Filter Usage ................................... [NONE DETECTED - PASS]
[CHECK 04] Ty

## 3. Automated Ghostscript Command Pipelines for Web Engines

Production conversion systems at [KaijuConverter](https://kaijuconverter.com) execute multi-stage Ghostscript transformation pipelines with isolated color calibration postscript initialization scripts (`PDFA_def.ps`).

### PDF/A-1b Ghostscript Invocation:
```bash
gs -dPDFA=1 -dBATCH -dNOPAUSE -dNOOUTERSAVE \
   -sProcessColorModel=DeviceRGB \
   -sColorConversionStrategy=UseDeviceIndependentColor \
   -sDEVICE=pdfwrite \
   -dPDFACompatibilityPolicy=1 \
   -dCompatibilityLevel=1.4 \
   -sOutputFile=document_archive_pdfa1b.pdf \
   PDFA_def.ps input_government_form.pdf
```

### PDF/A-3b Hybrid Invoicing Invocation (Factur-X / ZUGFeRD):
```bash
gs -dPDFA=3 -dBATCH -dNOPAUSE -dNOOUTERSAVE \
   -sProcessColorModel=DeviceRGB \
   -sColorConversionStrategy=UseDeviceIndependentColor \
   -sDEVICE=pdfwrite \
   -dPDFACompatibilityPolicy=1 \
   -dCompatibilityLevel=1.7 \
   -sOutputFile=invoice_facturx_compliant.pdf \
   PDFA_def.ps invoice_raw.pdf
```


## 4. Key Takeaways & Enterprise Recommendation

1. **Never rely on pure desktop 'Save as PDF'**: Most desktop PDF virtual printers fail to embed universal OutputIntent ICC tags, leaving the document non-compliant with European e-government gateways.
2. **Ensure Unicode font mappings**: Documents passing visual inspection may still fail veraPDF validation if ligatures or localized glyphs lack corresponding `/ToUnicode` mappings.
3. **For electronic invoicing, default to PDF/A-3b**: This guarantees backward-compatibility with PDF/A-2 readers while cleanly enclosing structured XML datasets (`factur-x.xml`) without corrupting document integrity.

--- 
For high-volume cloud processing, batch API conversions, or zero-retention document compliance, visit [KaijuConverter PDF Archival Tools](https://kaijuconverter.com/tools/pdf-to-pdfa).
